# 02. 공시 노출 라벨링 준비 (파일럿 P2)

S1 상위 5개 발행사의 10-K를 문단 단위로 추출하고, 사람이 정답 라벨을 붙일 시트를 만든다.

- 판정 기준: `labeling_guide.md` v1.1 (라벨링 전에 커밋되어 있어야 한다)
- 이 노트북은 **판정을 하지 않는다.** 키워드 적중 목록은 읽기 보조 도구다.
- LLM 추출은 이 노트북에 없다. 라벨 시트를 커밋한 **뒤에** 별도 노트북에서 실행한다.

**선행 조건:** `01_kic_13f_extraction.ipynb`를 실행해 `data/processed/S1/sample.csv`가 있어야 한다.

In [ ]:
# ── 설정 ─────────────────────────────────────────────
USE_FIXTURES = False   # True: 모의 응답 (01 노트북도 모의 모드로 먼저 실행해야 함)
USER_AGENT = "Blind Spot pilot Hyeseong Kim luckykhs@gachon.ac.kr"   # ← 01 노트북과 같게

TOP_N = 5              # 프로토콜 7.1: 상위 5개 발행사
LABELER = "Hyeseong Kim"   # 라벨 작성자 (labels_P2.csv 의 labeler 열)

In [ ]:
import sys, textwrap
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "tests")]
import kic13f as k
import filings as fl

if USE_FIXTURES:
    from fixtures import FakeClient
    client = FakeClient()
    S1_PATH = ROOT / "data" / "processed_fixture" / "S1" / "sample.csv"
    LABEL_DIR = ROOT / "data" / "labeling_fixture"
    print("⚠ 모의 응답 모드: 실제 공시가 아니다.")
else:
    client = k.SecClient(USER_AGENT, cache_dir=ROOT / "data" / "raw" / "sec_cache")
    S1_PATH = ROOT / "data" / "processed" / "S1" / "sample.csv"
    LABEL_DIR = ROOT / "data" / "labeling"

DOC_DIR = LABEL_DIR / "docs"
HIT_DIR = LABEL_DIR / "hits"
LABEL_PATH = LABEL_DIR / "labels_P2.csv"
for d in (DOC_DIR, HIT_DIR):
    d.mkdir(parents=True, exist_ok=True)

sample = pd.read_csv(S1_PATH).sort_values("rank")
sample[["rank", "cik", "edgar_name", "latest_10k_before_cutoff", "latest_10k_filing_date"]].head(TOP_N)

## 1. 10-K 원문 추출

각 발행사의 `latest_10k_before_cutoff` 10-K 본문을 문단으로 나누고, 문단마다 ID(`항목-p순번`)를 붙인다.
인용할 때 이 ID를 `evidence_location`에 적는다.

In [ ]:
docs = {}

def extract(row):
    url = fl.primary_document_url(client, row.cik, row.latest_10k_before_cutoff)
    df = fl.assign_items(fl.html_to_paragraphs(client.get_bytes(url)))
    df.to_csv(DOC_DIR / f"{row.cik}_{row.latest_10k_before_cutoff}.csv", index=False, encoding="utf-8-sig")
    docs[int(row.cik)] = df
    return url

for row in sample.head(TOP_N).itertuples(index=False):
    url = extract(row)
    print(f"[{row.rank}] {row.edgar_name:<28} 문단 {len(docs[row.cik]):>5}개  {url}")

### 항목 구분 점검

라벨링 범위(Item 1, 1A, 7과 재무제표 주석)가 잡혔는지 확인한다. 재무제표 주석은 회사에 따라
**Item 8 또는 Item 15**에 수록되므로(기준서 v1.1), `fin_notes_in` 열에서 실제 위치를 확인한다.
`ok = False`인 발행사는 `docs/*.csv`에서 해당 부분을 직접 찾아 읽는다(기준서 4절 2단계).

In [ ]:
coverage = pd.DataFrame([{"rank": row.rank, "company": row.edgar_name, **fl.coverage_check(docs[row.cik])}
                         for row in sample.head(TOP_N).itertuples(index=False)])
display(coverage)
print("⚠ 점검 필요:", coverage.loc[~coverage["ok"], "company"].tolist() or "없음")

## 2. 키워드 적중 목록 (읽기 보조)

테마별로 키워드가 들어 있는 문단을 모아 `hits/` 폴더에 저장한다. **적중했다고 노출이 있다는 뜻이 아니고,
적중하지 않았다고 노출이 없다는 뜻도 아니다.** 판정은 라벨링 기준서 3·4절을 따른다.

In [ ]:
counts = []
for row in sample.head(TOP_N).itertuples(index=False):
    rec = {"rank": row.rank, "company": row.edgar_name}
    for theme in ("T1", "T2a", "T2b"):
        hits = fl.keyword_hits(docs[row.cik], theme)
        hits.to_csv(HIT_DIR / f"r{row.rank}_{row.cik}_{theme}.csv", index=False, encoding="utf-8-sig")
        rec[theme] = len(hits)
    counts.append(rec)
pd.DataFrame(counts)

In [ ]:
def _wrap(text, width):
    # 하이픈·긴 단어에서 줄을 나누지 않는다. 화면에서 복사한 인용이 원문과 어긋나지 않게 하기 위함.
    return textwrap.fill(text, width=width, break_on_hyphens=False, break_long_words=False)

def show_hits(rank, theme, start=0, n=15, width=110):
    """적중 문단을 노트북 안에서 바로 읽는다. 예: show_hits(1, "T2b")"""
    row = sample[sample["rank"] == rank].iloc[0]
    hits = fl.keyword_hits(docs[int(row.cik)], theme)
    print(f"[{rank}] {row.edgar_name} / {theme}: 적중 {len(hits)}개 (표시 {start}~{min(start+n, len(hits))-1})\n")
    for h in hits.iloc[start:start + n].itertuples(index=False):
        print(f"── {h.pid}  [{h.terms}]")
        print(_wrap(h.text, width), "\n")

def show_paragraph(rank, pid, context=1, width=110):
    """특정 문단과 앞뒤 문단을 본다. 예: show_paragraph(1, "7-p01234")"""
    df = docs[int(sample[sample["rank"] == rank].iloc[0].cik)]
    i = df.index[df["pid"] == pid][0]
    for r in df.iloc[max(i - context, 0): i + context + 1].itertuples(index=False):
        print(f"── {r.pid}" + ("  ◀" if r.pid == pid else ""))
        print(_wrap(r.text, width), "\n")

def find_text(rank, pattern, items=None, n=20, width=110):
    """문서 전체에서 원하는 표현을 찾는다(대소문자 무관). 예: find_text(1, "Greater China|Hong Kong")"""
    df = docs[int(sample[sample["rank"] == rank].iloc[0].cik)]
    if items:
        df = df[df["item"].isin(items)]
    m = df[df["text"].str.contains(pattern, case=False, regex=True)]
    print(f"'{pattern}' 일치 {len(m)}개 (표시 최대 {n})\n")
    for r in m.head(n).itertuples(index=False):
        print(f"── {r.pid}")
        print(_wrap(r.text, width), "\n")

def raw(rank, pid):
    """문단 원문을 줄바꿈 없이 그대로 출력한다. 인용을 복사할 때 이 출력에서 복사한다."""
    df = docs[int(sample[sample["rank"] == rank].iloc[0].cik)]
    print(df.loc[df["pid"] == pid, "text"].iloc[0])

show_hits(1, "T1", n=5)

## 3. 라벨링 시트

`labels_P2.csv`를 만든다. **이미 있으면 덮어쓰지 않는다.**

작성 방법 (자세한 기준은 `labeling_guide.md`):
- `label`: `confirmed` / `possibly_related` / `not_found`
- `exposure_direction`: `positive` / `negative` / `mixed` / `unknown` (공시가 직접 말한 방향만)
- `evidence_text`: 원문 그대로 복사, 1~2문장
- `evidence_location`: 문단 ID (예: `7-p01234`)
- `searched`: not_found일 때 확인한 범위 (예: "적중 문단 전체 + Item 1A 정독")
- `labeler`, `labeled_at`: 작성자와 일시

Excel로 열어도 한글이 깨지지 않도록 UTF-8(BOM)으로 저장한다.

In [ ]:
if LABEL_PATH.exists():
    print("기존 시트를 유지한다:", LABEL_PATH.relative_to(ROOT))
else:
    fl.label_template(sample, TOP_N).to_csv(LABEL_PATH, index=False, encoding="utf-8-sig")
    print("새 시트를 만들었다:", LABEL_PATH.relative_to(ROOT))
pd.read_csv(LABEL_PATH, encoding="utf-8-sig", keep_default_na=False)

### 라벨 입력 도구

CSV를 Excel로 직접 고치지 않고, 아래 함수로 한 행씩 입력한다. 입력 즉시 허용 값과 인용 원문 일치를 검사한다.

```python
set_label(1, "T1", "confirmed", direction="positive", quote="원문 그대로", pid="7-p00123")
set_label(3, "T2a", "not_found", searched="적중 문단 전체 + 지역별 매출 주석 확인")
progress()   # 진행 현황
```

- 같은 행을 다시 입력하면 덮어쓴다(LLM 실행 전까지만 허용, 기준서 7절).
- `labels_P2.csv`를 Excel로 열어 둔 채 실행하면 Windows에서 저장이 실패할 수 있다. 확인할 때는 `progress()`를 쓴다.

In [ ]:
from datetime import datetime

def set_label(rank, theme, label, direction="unknown", quote="", pid="", searched="", note=""):
    assert label in fl.ALLOWED["label"], f"label 은 {sorted(fl.ALLOWED['label'])} 중 하나"
    assert direction in fl.ALLOWED["exposure_direction"] - {""}, "direction 은 positive/negative/mixed/unknown"
    if label == "not_found":
        assert not quote and searched, "not_found 는 quote 없이, searched(탐색 범위)를 적는다"
    else:
        assert quote and pid, "confirmed/possibly_related 는 quote 와 pid 가 필요하다"
    L = pd.read_csv(LABEL_PATH, encoding="utf-8-sig", keep_default_na=False)
    m = (L["rank"].astype(int) == rank) & (L["theme"] == theme)
    assert m.sum() == 1, f"rank {rank} / {theme} 행이 시트에 없다"
    if quote:
        v = fl.verify_quote(quote, docs[int(L.loc[m, "cik"].iloc[0])], pid)
        if v["status"] != "ok":
            print(f"✗ 저장하지 않음 — 인용 검증 {v['status']}" + (f" (실제 위치 {v['found_in']})" if v["found_in"] else ""))
            return
    L.loc[m, ["label", "exposure_direction", "evidence_text", "evidence_location", "searched", "note",
              "labeler", "labeled_at"]] = [label, direction, quote, pid, searched, note,
                                           LABELER, f"{datetime.now():%Y-%m-%d %H:%M}"]
    L.to_csv(LABEL_PATH, index=False, encoding="utf-8-sig")
    print(f"✓ 저장: rank {rank} / {theme} → {label}")

def progress():
    L = pd.read_csv(LABEL_PATH, encoding="utf-8-sig", keep_default_na=False)
    view = L.pivot_table(index=["rank", "company"], columns="theme", values="label", aggfunc="first")
    display(view.replace("", "·"))
    done = int((L["label"] != "").sum())
    print(f"완료 {done}/{len(L)}  |  분포 {L.loc[L['label'] != '', 'label'].value_counts().to_dict()}")

progress()

## 4. 라벨 검증

시트를 다 채운 뒤 실행한다. 모든 행이 `ok = True`가 될 때까지 시트를 고친다.
검증 항목: 허용 값, 인용 유무, **인용이 해당 문단에 원문 그대로 있는지**, not_found의 탐색 범위 기재.

In [ ]:
labels = pd.read_csv(LABEL_PATH, encoding="utf-8-sig", keep_default_na=False)
for cik in labels["cik"].astype(int).unique():      # 추가 쌍으로 들어온 발행사 문서도 준비
    if cik not in docs:
        extract(sample[sample["cik"] == cik].iloc[0])
check = fl.validate_labels(labels, docs)
display(check[~check["ok"]] if (~check["ok"]).any() else "모든 행 통과")
print(labels["label"].value_counts().to_dict())

### confirmed가 아닌 라벨이 4건 미만일 때 (프로토콜 7.1)

아래 셀을 한 번 실행할 때마다 다음 순서의 쌍(S1 rank 6의 T1 → T2a → T2b → rank 7 …)이 **한 개** 추가된다.
추가된 행을 라벨링한 뒤 다시 4절 검증을 실행하고, 4건 이상이 될 때까지 반복한다.

In [ ]:
labels = pd.read_csv(LABEL_PATH, encoding="utf-8-sig", keep_default_na=False)
non_confirmed = int((labels["label"].isin(["possibly_related", "not_found"])).sum())
unlabeled = int((labels["label"] == "").sum())
print(f"confirmed가 아닌 라벨 {non_confirmed}건, 미작성 {unlabeled}건")

if unlabeled:
    print("→ 먼저 모든 행을 라벨링한다.")
elif non_confirmed >= 4:
    print("→ 조건 충족. 추가 쌍 필요 없음.")
else:
    done = set(zip(labels["rank"].astype(int), labels["theme"]))
    nxt = next((r, t) for r in sample["rank"] for t in ("T1", "T2a", "T2b") if (r, t) not in done)
    row = sample[sample["rank"] == nxt[0]].iloc[0]
    new = pd.DataFrame([{"rank": nxt[0], "cik": row.cik, "company": row.edgar_name,
                         "theme": nxt[1], "added_by_rule": "yes"}]).reindex(columns=fl.LABEL_COLUMNS).fillna("")
    pd.concat([labels, new]).to_csv(LABEL_PATH, index=False, encoding="utf-8-sig")
    if int(row.cik) not in docs:
        extract(row)
    print(f"→ 추가: rank {nxt[0]} {row.edgar_name} / {nxt[1]} — 라벨링 후 4절 검증을 다시 실행")

## 5. 커밋

검증을 모두 통과하면 **LLM 추출을 실행하기 전에** 라벨 시트를 커밋한다.

```bash
git add data/labeling/labels_P2.csv
git commit -m "data: human gold labels for P2 (before LLM extraction)"
git push
```

`experiment_log.md`에 라벨링 완료 일시, 라벨 분포, 추가 쌍 여부를 한 줄로 남긴다.